# Early Sepsis Prediction with a CNN-LSTM: Tuned Pipeline (Run 2)

This is the second run of the corrected pipeline. Everything up to the model (data checks, the patient split, the causal features, the leakage check, preprocessing) is identical to Run 1, and so is the evaluation. **The only change is how the CNN-LSTM is trained.**

### Why a second run
In Run 1, all three CNN-LSTM seeds reached their best validation score after **epoch 1**, then got steadily worse (validation AUROC 0.82 → 0.75) while the training loss kept falling. The network memorised the 2,052 septic training patients almost immediately, so its test score (AUROC 0.826) was really a first-epoch score.

### What Run 2 changes
* **Validation 4× per pass** over the training data, so early stopping can find the best point *inside* the first pass instead of only after it.
* **A small hyperparameter search** (10 configurations, one seed each): lower learning rates, stronger weight decay and dropout, input noise, a smaller network, all negatives per pass, and a 48-hour window. Each configuration is scored on **validation AUPRC only**.
* **A larger final ensemble**: 5 seeds of the winning configuration.
* **Same test patients as Run 1**, verified in Step 4, so the Run 1 vs Run 2 comparison at the end is like-for-like.

Choosing among 10 configurations on validation makes the *validation* scores slightly optimistic. The test set is still untouched until Step 14, so the test numbers stay unbiased.

---
### Original fixes carried over from Run 1

### What this notebook fixes (and why the old models were stuck at AUROC ~0.77, F1 ~0.12)

| Problem in the old pipeline | Fix here |
|---|---|
| `ICULOS` dropped as "just an index" — it is one of the strongest predictors | Kept, plus `log(ICULOS)` |
| Lab values only taken as the median inside the 10-h window → most lab values missing | Labs carried forward per patient (last observation): missing lab values drop from 95 % (raw hours) to 49 % (41 % across all 34 physiological variables) |
| Missing values replaced by π (= "3 SD abnormal" after scaling) | Missingness encoded explicitly (`n_measurements`, `hours_since_last`), remaining gaps set to 0 (= mean) |
| `bfill()` used future values (leakage) | Strictly causal features; an automated check proves it (Step 6) |
| Validation set = random *windows* (same patient in train and val) | Patient-level train / val / test split, stratified by sepsis and hospital |
| Keras `AUC` on a 2-column softmax reported ~0.96–0.99 while the true AUROC was ~0.78 | Single sigmoid output; validation AUROC/AUPRC computed with scikit-learn every epoch |
| Evaluated the last-epoch weights, not the saved best ones | `restore_best_weights=True`, models saved after restoring |
| Threshold tuned on the test set | All thresholds chosen on validation only, then applied once to test |
| SMOTE on π-filled vectors + class weight + focal loss (triple correction) | Plain BCE + fresh random negative subsampling each epoch, probabilities re-calibrated afterwards |
| First 10 h of every stay never predicted | Every ICU hour is predicted (windows are zero-padded at the start of a stay) |

### Pipeline overview
1. Setup & configuration
2. Load and sanity-check the data
3. Exploratory analysis
4. Patient-level split (**test set is written to disk immediately and never touched until Step 14**)
5. Causal feature engineering
6. Leakage check (features at hour *t* must not change when later hours are removed)
7. Preprocessing fitted on the training patients only
8. Sequence windows (last 24 h for every hour)
9. CNN-LSTM model
10. Hyperparameter search on validation, then final training (5 seeds → ensemble)
11. Gradient-boosting baseline (LightGBM)
12. Probability calibration & model blending (validation only)
13. Threshold selection (validation only)
14. Final evaluation on the test set (with bootstrap 95 % CIs)
15. Early-warning / patient-level analysis
16. Interpretability (permutation importance + SHAP)
17. Export artifacts & re-load check

### How to run on Kaggle
* Add your dataset (the PhysioNet 2019 `Dataset.csv`) as an input — the path is detected automatically.
* **Settings → Accelerator → GPU T4** (recent TensorFlow builds may not support the older P100). Internet is not required.
* *Run All*. Expected runtime: roughly 2–3.5 h with a GPU (mostly the Step 10 search).
* Optional: if you add Run 1's `split_patient_ids.json` as an input, it is loaded directly. Otherwise the split is recomputed (same seed, same data → same patients) and checked against Run 1.
* Download `/kaggle/working/sepsis_artifacts.zip` at the end (also visible in the Output tab).

## 1. Setup & configuration

In [ ]:
import os, gc, sys, json, glob, time, math, random, shutil, warnings, importlib
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix, brier_score_loss)
from sklearn.calibration import calibration_curve

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import tensorflow as tf
import keras
from keras import layers

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
sns.set_theme(style='whitegrid', context='notebook')

print('Python     ', sys.version.split()[0])
print('TensorFlow ', tf.__version__)
print('Keras      ', keras.__version__)
print('GPUs       ', tf.config.list_physical_devices('GPU'))

In [ ]:
# ---------------------------------------------------------------- configuration
SMOKE_TEST = os.environ.get('SEPSIS_SMOKE', '0') == '1'   # quick tiny run for debugging; leave False on Kaggle

CFG = dict(
    SEED=42,
    TEST_FRAC=0.15,             # fraction of PATIENTS held out for the final test
    VAL_FRAC=0.15,              # fraction of PATIENTS used for early stopping / thresholds / blending
    BATCH_SIZE=512,
    PRED_BATCH_SIZE=2048,
    EVALS_PER_PASS=4,           # validation checks per pass over the training data (Run 1 peaked inside pass 1)
    MAX_PASSES=12,              # hard cap on passes over the training data per model
    PATIENCE=8,                 # early stopping patience in validation checks (8 checks = 2 passes)
    LR_PATIENCE=3,              # halve the learning rate after 3 checks without improvement
    RUN_SWEEP=True,             # False = skip the search and use BEST_HP_OVERRIDE (Step 10)
    N_SEEDS=5,                  # final ensemble size for the selected configuration
    TARGET_SENSITIVITY=0.80,    # for the "high-sensitivity" operating point
    N_BOOTSTRAP=200,            # bootstrap resamples (by patient) for 95 % confidence intervals
    WORKERS=2,
)
if SMOKE_TEST:
    CFG.update(MAX_PASSES=1, EVALS_PER_PASS=2, PATIENCE=2, N_SEEDS=1, N_BOOTSTRAP=20, SMOKE_N_PATIENTS=1500)

WORK_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.abspath('./sepsis_run')
ART_DIR = os.path.join(WORK_DIR, 'artifacts')
PLOT_DIR = os.path.join(ART_DIR, 'plots')
MODEL_DIR = os.path.join(ART_DIR, 'models')
for d in (ART_DIR, PLOT_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)
os.chdir(WORK_DIR)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)


def savefig(name):
    path = os.path.join(PLOT_DIR, name)
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.show()
    plt.close()


set_seed(CFG['SEED'])
print('Smoke test:', SMOKE_TEST)
print('Outputs   :', ART_DIR)
print(json.dumps(CFG, indent=1))

### Shared pipeline module
All feature engineering, preprocessing, windowing and the PhysioNet utility score live in one module, `sepsis_pipeline.py`. The notebook writes it to disk and imports it, and it is exported with the model, so the **backend can use exactly the same code** to turn raw hourly patient data into model inputs.

Every feature at hour *t* uses only data from hours ≤ *t* of the same patient. Nothing is computed across patients, so engineering features on the full dataset before splitting cannot leak information.

In [ ]:
%%writefile sepsis_pipeline.py
"""Causal feature engineering, preprocessing, windowing and evaluation utilities
for hourly ICU sepsis prediction (PhysioNet/CinC 2019 format).

Used for BOTH training and inference so the model always sees identical features.
Every feature at hour t depends only on hours <= t of the same patient.
"""
import json
import warnings
import numpy as np
import pandas as pd

VITALS = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2']
LABS = ['BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos',
        'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate',
        'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb',
        'PTT', 'WBC', 'Fibrinogen', 'Platelets']
PHYSIO = VITALS + LABS
DEMOGRAPHICS = ['Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime']
RAW_COLUMNS = PHYSIO + DEMOGRAPHICS + ['ICULOS', 'Patient_ID']
LABEL = 'SepsisLabel'

ROLL_VARS = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp']   # short-term dynamics
LONG_VARS = ['HR', 'Temp', 'Resp', 'MAP', 'O2Sat', 'SBP']           # deviation from own baseline
ROLL_WIN = 6
LONG_WIN = 24
HRS_SINCE_CAP = 72.0
F32 = np.float32


# ------------------------------------------------------------------ loading
def prepare_raw(df):
    """Standardise a raw hourly table: column set, dtypes, sort order."""
    df = df.copy()
    df = df.drop(columns=[c for c in df.columns if c.startswith('Unnamed') or c == 'Hour'])
    if 'Patient_ID' not in df.columns or 'ICULOS' not in df.columns:
        raise ValueError('Input needs at least Patient_ID and ICULOS columns')
    df['Patient_ID'] = df['Patient_ID'].astype(str)
    for c in PHYSIO + DEMOGRAPHICS:
        if c not in df.columns:
            df[c] = np.nan
    num = PHYSIO + DEMOGRAPHICS + ['ICULOS']
    df[num] = df[num].apply(pd.to_numeric, errors='coerce').astype(F32)
    if LABEL in df.columns:
        df[LABEL] = df[LABEL].fillna(0).astype(np.int8)
    df = df.sort_values(['Patient_ID', 'ICULOS'], kind='mergesort').reset_index(drop=True)
    return df


def load_raw(path, nrows=None):
    return prepare_raw(pd.read_csv(path, dtype={'Patient_ID': str}, nrows=nrows))


# ------------------------------------------------------------------ helpers
def patient_codes(df):
    """0..n_patients-1, contiguous; df must be sorted by Patient_ID."""
    return pd.factorize(df['Patient_ID'])[0]


def lag(arr, k, codes):
    """Value k rows earlier within the same patient (NaN across patient boundary)."""
    out = np.full(arr.shape, np.nan, dtype=F32)
    if k >= len(arr):
        return out
    out[k:] = arr[:-k]
    same = np.zeros(len(arr), dtype=bool)
    same[k:] = codes[k:] == codes[:-k]
    out[~same] = np.nan
    return out


def _rolling_stack(v, win, codes):
    return np.stack([v] + [lag(v, k, codes) for k in range(1, win)])


def _f(x):
    return np.asarray(x, dtype=F32)


# ------------------------------------------------------------------ features
def engineer_features(df):
    """df: output of prepare_raw. Returns (X: DataFrame float32, meta: DataFrame)."""
    codes = patient_codes(df)
    icu = df['ICULOS'].to_numpy(F32)
    feats = {}

    # 1) last observation carried forward + measurement-process features
    ff = df.groupby(codes, sort=False)[PHYSIO].ffill()
    for c in PHYSIO:
        measured = df[c].notna().to_numpy()
        feats[c] = ff[c].to_numpy(F32)
        n_meas = pd.Series(measured.astype(F32)).groupby(codes).cumsum().to_numpy()
        feats[f'{c}__log_nmeas'] = _f(np.log1p(n_meas))
        last_t = pd.Series(np.where(measured, icu, np.nan)).groupby(codes).ffill().to_numpy()
        since = icu - last_t
        feats[f'{c}__hrs_since'] = _f(np.where(np.isnan(since), HRS_SINCE_CAP,
                                               np.minimum(since, HRS_SINCE_CAP)))

    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        # 2) short-term dynamics (last 6 h)
        for c in ROLL_VARS:
            v = feats[c]
            st = _rolling_stack(v, ROLL_WIN, codes)
            feats[f'{c}__mean6'] = _f(np.nanmean(st, 0))
            feats[f'{c}__min6'] = _f(np.nanmin(st, 0))
            feats[f'{c}__max6'] = _f(np.nanmax(st, 0))
            feats[f'{c}__std6'] = _f(np.nanstd(st, 0))
            feats[f'{c}__d1'] = v - lag(v, 1, codes)
            feats[f'{c}__d6'] = v - lag(v, 6, codes)
        # 3) deviation from the patient's own recent (24 h) and whole-stay baseline
        for c in LONG_VARS:
            v = feats[c]
            st = _rolling_stack(v, LONG_WIN, codes)
            feats[f'{c}__dev24'] = v - _f(np.nanmean(st, 0))
            ok = ~np.isnan(v)
            s = pd.Series(np.where(ok, v, 0.0)).groupby(codes).cumsum().to_numpy()
            n = pd.Series(ok.astype(F32)).groupby(codes).cumsum().to_numpy()
            feats[f'{c}__dev_base'] = _f(v - np.where(n > 0, s / np.maximum(n, 1), np.nan))

        # 4) clinical scores (computed on carried-forward values; NaN comparisons -> False)
        hr, sbp, dbp, mp = feats['HR'], feats['SBP'], feats['DBP'], feats['MAP']
        resp, temp, wbc = feats['Resp'], feats['Temp'], feats['WBC']
        plt_, bili, creat = feats['Platelets'], feats['Bilirubin_total'], feats['Creatinine']
        feats['shock_index'] = _f(np.where(sbp > 0, hr / sbp, np.nan))
        feats['pulse_pressure'] = sbp - dbp
        feats['bun_creat_ratio'] = _f(np.where(creat > 0, feats['BUN'] / creat, np.nan))
        feats['sirs'] = _f((hr > 90).astype(int) + ((temp > 38) | (temp < 36))
                           + ((resp > 20) | (feats['PaCO2'] < 32)) + ((wbc > 12) | (wbc < 4)))
        feats['qsofa_partial'] = _f((resp >= 22).astype(int) + (sbp <= 100))
        sofa = ((plt_ < 150).astype(int) + (plt_ < 100) + (plt_ < 50) + (plt_ < 20)
                + (bili >= 1.2) + (bili >= 2) + (bili >= 6) + (bili >= 12)
                + (creat >= 1.2) + (creat >= 2) + (creat >= 3.5) + (creat >= 5)
                + (mp < 70))
        feats['sofa_partial'] = _f(sofa)
        feats['sofa_partial__d24'] = feats['sofa_partial'] - lag(feats['sofa_partial'], 24, codes)
        feats['lactate_gt2'] = _f(feats['Lactate'] > 2)
        feats['map_lt65'] = _f(mp < 65)

    # 5) demographics / admission / time
    u1, u2 = df['Unit1'].to_numpy(F32), df['Unit2'].to_numpy(F32)
    feats['Age'] = df['Age'].to_numpy(F32)
    feats['Gender'] = df['Gender'].to_numpy(F32)
    feats['Unit1'] = _f(np.nan_to_num(u1))
    feats['Unit2'] = _f(np.nan_to_num(u2))
    feats['unit_unknown'] = _f(np.isnan(u1) & np.isnan(u2))
    feats['HospAdmTime'] = df['HospAdmTime'].to_numpy(F32)
    feats['ICULOS'] = icu
    feats['log_ICULOS'] = _f(np.log1p(np.maximum(icu, 0)))

    X = pd.DataFrame(feats)
    X = X.replace([np.inf, -np.inf], np.nan).astype(F32)
    meta = pd.DataFrame({'Patient_ID': df['Patient_ID'].to_numpy(),
                         'ICULOS': df['ICULOS'].to_numpy(),
                         'hour_idx': pd.Series(codes).groupby(codes).cumcount().to_numpy()})
    if LABEL in df.columns:
        meta[LABEL] = df[LABEL].to_numpy()
    return X, meta


def feature_group(name):
    """Base clinical variable a feature belongs to (for grouped importance)."""
    return name.split('__')[0]


# ------------------------------------------------------------------ preprocessing
class Preprocessor:
    """Clip to training-set 0.5/99.5 percentiles, standardise, fill remaining NaN with 0 (= mean)."""

    def fit(self, X, q_low=0.5, q_high=99.5):
        self.columns = list(X.columns)
        self.lo, self.hi, self.mean, self.std = {}, {}, {}, {}
        for c in self.columns:
            v = X[c].to_numpy(np.float64)
            v = v[~np.isnan(v)]
            if len(v) == 0:
                lo = hi = mu = 0.0
                sd = 1.0
            else:
                lo, hi = np.percentile(v, [q_low, q_high])
                v = np.clip(v, lo, hi)
                mu, sd = float(v.mean()), float(v.std())
                sd = sd if sd > 1e-6 else 1.0
            self.lo[c], self.hi[c], self.mean[c], self.std[c] = float(lo), float(hi), float(mu), float(sd)
        return self

    def transform(self, X):
        out = np.empty((len(X), len(self.columns)), dtype=F32)
        for j, c in enumerate(self.columns):
            v = X[c].to_numpy(F32) if c in X.columns else np.full(len(X), np.nan, F32)
            v = (np.clip(v, self.lo[c], self.hi[c]) - self.mean[c]) / self.std[c]
            out[:, j] = np.nan_to_num(v, nan=0.0)
        return out

    def to_dict(self):
        return dict(columns=self.columns, lo=self.lo, hi=self.hi, mean=self.mean, std=self.std)

    @classmethod
    def from_dict(cls, d):
        p = cls()
        p.columns, p.lo, p.hi, p.mean, p.std = d['columns'], d['lo'], d['hi'], d['mean'], d['std']
        return p

    def save(self, path):
        with open(path, 'w') as f:
            json.dump(self.to_dict(), f)

    @classmethod
    def load(cls, path):
        with open(path) as f:
            return cls.from_dict(json.load(f))


# ------------------------------------------------------------------ windows
def build_padded(Xs, codes, seq_len):
    """Insert seq_len-1 zero rows before every patient and add an 'is_real' channel.
    Window for row i is P[starts[i] : starts[i] + seq_len] (ends at row i, never crosses patients)."""
    n, f = Xs.shape
    n_pat = int(codes.max()) + 1 if n else 0
    P = np.zeros((n + n_pat * (seq_len - 1), f + 1), dtype=F32)
    pos = np.arange(n) + (codes + 1) * (seq_len - 1)
    P[pos, :f] = Xs
    P[pos, f] = 1.0
    starts = pos - (seq_len - 1)
    return P, starts


def gather_windows(P, starts, seq_len):
    return P[starts[:, None] + np.arange(seq_len)[None, :]]


def calibrate(p, log_offset):
    """Undo the effect of negative subsampling: logit_true = logit_model + log(neg_fraction)."""
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-7, 1 - 1e-7)
    z = np.log(p / (1 - p)) + log_offset
    return 1 / (1 + np.exp(-z))


def predict_windows(models, P, starts, seq_len, batch_size=2048):
    """Average probability of several Keras models over all windows."""
    out = np.zeros(len(starts), dtype=np.float64)
    for m in models:
        for b in range(0, len(starts), batch_size):
            xb = gather_windows(P, starts[b:b + batch_size], seq_len)
            out[b:b + batch_size] += np.asarray(m.predict_on_batch(xb)).ravel()
    return out / max(len(models), 1)


def predict_raw(raw_df, models, prep, config):
    """Raw hourly table -> per-hour calibrated sepsis probability (same order as meta)."""
    df = prepare_raw(raw_df)
    X, meta = engineer_features(df)
    Xs = prep.transform(X)
    P, starts = build_padded(Xs, patient_codes(df), config['seq_len'])
    p = predict_windows(models, P, starts, config['seq_len'])
    meta['p_cnn_lstm'] = calibrate(p, config['calibration_log_offset'])
    return meta


# ------------------------------------------------------------------ PhysioNet 2019 utility
class UtilityScorer:
    """Normalised utility score of the PhysioNet/CinC 2019 challenge, vectorised.
    meta must be sorted by patient/time and contain Patient_ID, hour_idx, SepsisLabel."""

    def __init__(self, meta, dt_early=-12, dt_optimal=-6, dt_late=3, max_u_tp=1.0, min_u_fn=-2.0, u_fp=-0.05):
        codes = patient_codes(meta)
        lab = meta[LABEL].to_numpy()
        hour = meta['hour_idx'].to_numpy().astype(np.float64)
        first = pd.Series(np.where(lab == 1, hour, np.inf)).groupby(codes).transform('min').to_numpy()
        septic = np.isfinite(first)
        t_sepsis = first - dt_optimal
        dt = np.where(septic, hour - t_sepsis, 0.0)
        m1, b1 = max_u_tp / (dt_optimal - dt_early), -max_u_tp / (dt_optimal - dt_early) * dt_early
        m2 = -max_u_tp / (dt_late - dt_optimal); b2 = -m2 * dt_late
        m3 = min_u_fn / (dt_late - dt_optimal); b3 = -m3 * dt_optimal
        u1_sep = np.where(dt <= dt_optimal, np.maximum(m1 * dt + b1, u_fp),
                          np.where(dt <= dt_late, m2 * dt + b2, 0.0))
        u0_sep = np.where(dt <= dt_optimal, 0.0, np.where(dt <= dt_late, m3 * dt + b3, 0.0))
        self.u1 = np.where(septic, u1_sep, u_fp)
        self.u0 = np.where(septic, u0_sep, 0.0)
        best_pred = septic & (dt >= dt_early) & (dt <= dt_late)
        self.best_row = np.where(best_pred, self.u1, self.u0)
        self.best = float(self.best_row.sum())
        self.inaction = float(self.u0.sum())

    def score(self, pred):
        obs = float(np.where(pred, self.u1, self.u0).sum())
        return (obs - self.inaction) / (self.best - self.inaction)

In [ ]:
if 'sepsis_pipeline' in sys.modules:
    importlib.reload(sys.modules['sepsis_pipeline'])
import sepsis_pipeline as sp
print('Pipeline module loaded:', sp.__file__)

## 2. Load and sanity-check the data

In [ ]:
# >>> Path to your dataset on Kaggle. Change this line if the file moves. <<<
KAGGLE_DATA_PATH = '/kaggle/input/datasets/jerrymardeburg/my-dataset/Dataset.csv'


def find_dataset():
    env = os.environ.get('SEPSIS_CSV')
    if env:
        return env
    if os.path.isfile(KAGGLE_DATA_PATH):
        return KAGGLE_DATA_PATH
    print(f'{KAGGLE_DATA_PATH} not found - searching /kaggle/input for the sepsis CSV...')
    hits = sorted(glob.glob('/kaggle/input/**/*.csv', recursive=True)) + sorted(glob.glob('./*.csv'))
    for h in hits:
        cols = set(pd.read_csv(h, nrows=0).columns)
        if {'SepsisLabel', 'Patient_ID', 'ICULOS'} <= cols:
            return h
    raise FileNotFoundError('Could not find the sepsis CSV. Add it as a Kaggle input or set SEPSIS_CSV.')


DATA_PATH = find_dataset()
print('Dataset:', DATA_PATH)

t0 = time.time()
raw = sp.load_raw(DATA_PATH, nrows=CFG['SMOKE_N_PATIENTS'] * 60 if SMOKE_TEST else None)
if SMOKE_TEST:
    keep = raw['Patient_ID'].unique()[:CFG['SMOKE_N_PATIENTS']]
    raw = raw[raw['Patient_ID'].isin(keep)].reset_index(drop=True)
print(f'Loaded {len(raw):,} rows x {raw.shape[1]} columns in {time.time() - t0:.0f}s')
raw.head()

In [ ]:
n_pat = raw['Patient_ID'].nunique()
per_pat = raw.groupby('Patient_ID')
stay_len = per_pat.size()
septic_pat = per_pat['SepsisLabel'].max()

# integrity checks
icu_increasing = per_pat['ICULOS'].apply(lambda s: s.is_monotonic_increasing and s.is_unique).all()
label_monotone = per_pat['SepsisLabel'].apply(lambda s: s.is_monotonic_increasing).all()
dup_rows = raw.duplicated(['Patient_ID', 'ICULOS']).sum()

print(f'Patients                 : {n_pat:,}')
print(f'Hourly rows              : {len(raw):,}')
print(f'Stay length (h)          : median {stay_len.median():.0f}, IQR {stay_len.quantile(.25):.0f}-{stay_len.quantile(.75):.0f}, max {stay_len.max()}')
print(f'Septic patients          : {septic_pat.sum():,} ({septic_pat.mean():.2%})')
print(f'Positive hours           : {raw.SepsisLabel.sum():,} ({raw.SepsisLabel.mean():.2%})')
print(f'ICULOS strictly increasing per patient : {icu_increasing}')
print(f'Label never goes 1 -> 0 per patient    : {label_monotone}')
print(f'Duplicate (patient, hour) rows         : {dup_rows}')

**About the label.** In the PhysioNet 2019 data, `SepsisLabel` is already shifted: it becomes 1 six hours *before* the clinical onset time (t_sepsis − 6 h). Predicting `SepsisLabel` at hour *t* from data up to hour *t* therefore means **predicting sepsis about 6 hours before onset**. This is the standard formulation of the challenge, and it is what this notebook does at every ICU hour.

## 3. Exploratory analysis

In [ ]:
first_pos = raw[raw.SepsisLabel == 1].groupby('Patient_ID')['ICULOS'].min()
hospital = pd.Series(np.where(pd.to_numeric(stay_len.index, errors='coerce') >= 100000, 'B', 'A'),
                     index=stay_len.index)

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
ax[0].hist(stay_len.clip(upper=200), bins=60, color='#4C72B0')
ax[0].set(title='ICU stay length (h, clipped at 200)', xlabel='hours', ylabel='patients')
ax[1].hist(first_pos.clip(upper=200), bins=60, color='#C44E52')
ax[1].set(title='First positive label hour (septic patients)', xlabel='ICULOS', ylabel='patients')
miss = raw[sp.PHYSIO].isna().mean().sort_values()
ax[2].barh(miss.index, miss.values * 100, color='#55A868')
ax[2].set(title='% of hours with no measurement', xlabel='% missing')
ax[2].tick_params(axis='y', labelsize=7)
plt.tight_layout()
savefig('01_eda_overview.png')

print(f'Septic patients already positive within first 10 h: {(first_pos <= 10).mean():.1%}')
print('Patients by hospital:', hospital.value_counts().to_dict())
print('Sepsis rate by hospital:', septic_pat.groupby(hospital).mean().round(4).to_dict())

## 4. Patient-level train / validation / test split
Patients, not rows or windows, are assigned to exactly one split, stratified by *ever septic* × *hospital*, so overlapping windows from one patient can never sit on both sides.

* **Train (70 %)**: model fitting.
* **Validation (15 %)**: early stopping, calibration, blending weights, thresholds.
* **Test (15 %)**: used **once**, in Step 14.

The raw rows of the test patients are written to `test_patients_raw.csv` right away. This file has the original hourly format, so you can re-test the exported model later or feed it to the web app.

In [ ]:
pat_info = pd.DataFrame({'septic': septic_pat, 'hospital': hospital})
strata = pat_info['septic'].astype(str) + '_' + pat_info['hospital']
if strata.value_counts().min() < 4:      # only relevant for tiny smoke-test samples
    strata = pat_info['septic'].astype(str)

prev_split = os.environ.get('SEPSIS_SPLIT_FILE') or next(
    iter(sorted(glob.glob('/kaggle/input/**/split_patient_ids.json', recursive=True))), None)
if prev_split and not SMOKE_TEST:
    with open(prev_split) as f:
        SPLIT_IDS = json.load(f)
    train_ids, val_ids, test_ids = (np.asarray(SPLIT_IDS[k]) for k in ('train', 'val', 'test'))
    print('Loaded the patient split from Run 1:', prev_split)
else:
    trval_ids, test_ids = train_test_split(pat_info.index.to_numpy(), test_size=CFG['TEST_FRAC'],
                                           stratify=strata, random_state=CFG['SEED'])
    train_ids, val_ids = train_test_split(trval_ids, test_size=CFG['VAL_FRAC'] / (1 - CFG['TEST_FRAC']),
                                          stratify=strata.loc[trval_ids], random_state=CFG['SEED'])
    SPLIT_IDS = {'train': sorted(train_ids), 'val': sorted(val_ids), 'test': sorted(test_ids)}
    print('Recomputed the patient split with the same seed as Run 1.')
assert not (set(train_ids) & set(val_ids)) and not (set(train_ids) & set(test_ids)) and not (set(val_ids) & set(test_ids))

with open(os.path.join(ART_DIR, 'split_patient_ids.json'), 'w') as f:
    json.dump(SPLIT_IDS, f)

# untouched raw test data, original hourly format
test_raw_path = os.path.join(ART_DIR, 'test_patients_raw.csv')
raw[raw.Patient_ID.isin(test_ids)].to_csv(test_raw_path, index=False)

summary = []
for name, ids in SPLIT_IDS.items():
    r = raw[raw.Patient_ID.isin(ids)]
    summary.append(dict(split=name, patients=len(ids), septic_patients=int(pat_info.loc[ids, 'septic'].sum()),
                        septic_rate=pat_info.loc[ids, 'septic'].mean(), hours=len(r), positive_hour_rate=r.SepsisLabel.mean()))
split_summary = pd.DataFrame(summary)
split_summary.to_csv(os.path.join(ART_DIR, 'split_summary.csv'), index=False)

# Run 1 fingerprint: hours per split. Identical counts = identical patients (same seed, same data).
RUN1_SPLIT_HOURS = {'train': 1087335, 'val': 231692, 'test': 233183}
SPLIT_MATCHES_RUN1 = dict(zip(split_summary['split'], split_summary['hours'].astype(int))) == RUN1_SPLIT_HOURS
if SMOKE_TEST:
    print('(smoke test - split check against Run 1 skipped)')
elif SPLIT_MATCHES_RUN1:
    print('Split check PASSED: same train/val/test patients as Run 1, so the results are directly comparable.')
else:
    print('WARNING: the split differs from Run 1. All comparisons INSIDE this notebook stay valid, '
          'but do not compare its test numbers with Run 1.')
print('Saved raw test patients ->', test_raw_path)
split_summary

## 5. Causal feature engineering
For every patient-hour (all features use only past and present data):

| Group | Features |
|---|---|
| Physiology (34 variables) | last observed value carried forward |
| Measurement process | `log(1 + number of measurements so far)`, `hours since last measurement` (capped at 72 h). Clinicians order labs such as lactate when they suspect sepsis, so *whether* something was measured is itself informative. |
| Short-term dynamics (7 vitals) | 6-h mean / min / max / std, 1-h and 6-h change |
| Personal baseline (6 vitals) | deviation from the patient's 24-h mean and from their whole-stay mean |
| Clinical scores | shock index, pulse pressure, BUN/creatinine, SIRS count, partial qSOFA, partial SOFA (+ 24-h change), lactate > 2, MAP < 65 |
| Context | age, gender, ICU unit, hospital-admission time, `ICULOS`, `log(ICULOS)` |

In [ ]:
t0 = time.time()
X_all, meta_all = sp.engineer_features(raw)
FEATURES = list(X_all.columns)
assert (meta_all['Patient_ID'].to_numpy() == raw['Patient_ID'].to_numpy()).all()
print(f'Feature matrix: {X_all.shape[0]:,} rows x {X_all.shape[1]} features  ({time.time() - t0:.0f}s, '
      f'{X_all.memory_usage().sum() / 1e9:.2f} GB)')

split_of = pd.Series('train', index=raw.Patient_ID.unique())
split_of.loc[SPLIT_IDS['val']] = 'val'
split_of.loc[SPLIT_IDS['test']] = 'test'
row_split = meta_all['Patient_ID'].map(split_of).to_numpy()
masks = {s: row_split == s for s in ('train', 'val', 'test')}

pd.Series({g: sum(sp.feature_group(f) == g for f in FEATURES) for g in dict.fromkeys(map(sp.feature_group, FEATURES))}).to_frame('n_features').T

## 6. Leakage check
For a random sample of patients, all data after a random cut-off hour is deleted and the features are recomputed. If any feature at the cut-off hour changes, it was using future information. This check must pass; it supports the "no look-ahead" claim in the paper.

In [ ]:
rng = np.random.default_rng(CFG['SEED'])
long_pats = stay_len[stay_len >= 30].index.to_numpy()
sample_pats = rng.choice(long_pats, size=min(40, len(long_pats)), replace=False)
pieces, cut_rows = [], []
for p in sample_pats:
    r = raw[raw.Patient_ID == p]
    cut = int(rng.integers(5, len(r) - 1))
    pieces.append(r.iloc[:cut])
    cut_rows.append(r.index[cut - 1])
trunc = sp.prepare_raw(pd.concat(pieces))
X_trunc, meta_trunc = sp.engineer_features(trunc)
last_rows = meta_trunc.groupby('Patient_ID').tail(1).index
full_vals = X_all.loc[cut_rows].reset_index(drop=True)
trunc_vals = X_trunc.loc[last_rows].reset_index(drop=True)
# align by patient id
order_full = meta_all.loc[cut_rows, 'Patient_ID'].to_numpy()
order_trunc = meta_trunc.loc[last_rows, 'Patient_ID'].to_numpy()
trunc_vals = trunc_vals.set_index(order_trunc).loc[order_full].reset_index(drop=True)
same = np.isclose(full_vals.to_numpy(), trunc_vals.to_numpy(), equal_nan=True, atol=1e-4)
bad = [FEATURES[j] for j in np.where(~same.all(0))[0]]
assert not bad, f'Features using future information: {bad}'
print(f'PASSED: {len(sample_pats)} patients, {len(FEATURES)} features - no feature uses future data.')

## 7. Preprocessing (fitted on training patients only)
Each feature is clipped to the training 0.5th–99.5th percentile, standardised with the training mean and SD, and any remaining gaps are set to 0 (the training mean). Gaps that remain are the hours before a variable is first measured, and the `hrs_since` / `n_meas` features already tell the model about them.

In [ ]:
t0 = time.time()
prep = sp.Preprocessor().fit(X_all[masks['train']])
Xs_all = prep.transform(X_all)
prep.save(os.path.join(ART_DIR, 'preprocessor.json'))
print(f'Preprocessor fitted on {masks["train"].sum():,} training rows ({time.time() - t0:.0f}s)')
print('Any NaN/inf after transform:', not np.isfinite(Xs_all).all())

## 8. Sequence windows
The CNN-LSTM at hour *t* sees the last `SEQ_LEN` hours (24 or 48; the search in Step 10 chooses), each with the full feature vector plus an `is_real` flag. Early in a stay, hours before admission are zero-padded with `is_real = 0`, so **every** ICU hour gets a prediction, including the first hours, where 30 % of septic patients already turn positive.

Windows are cut on the fly from one padded matrix per split and window length, so copies of every window are never stored in memory. During training, one pass over the data is split into `EVALS_PER_PASS` chunks, with a validation check after each chunk.

In [ ]:
SEQ_LENS = (24, 48)


def make_split(name, seq_len):
    m = masks[name]
    meta = meta_all[m].reset_index(drop=True)
    codes = sp.patient_codes(meta)
    P, starts = sp.build_padded(Xs_all[m], codes, seq_len)
    return dict(name=name, P=P, starts=starts, meta=meta, seq_len=seq_len,
                y=meta['SepsisLabel'].to_numpy().astype(np.float32))


DATA_BY_LEN = {L: {s: make_split(s, L) for s in ('train', 'val', 'test')} for L in SEQ_LENS}
N_CH = DATA_BY_LEN[SEQ_LENS[0]]['train']['P'].shape[1]
del Xs_all
gc.collect()
for L, data in DATA_BY_LEN.items():
    for s, d in data.items():
        print(f'{L}h {s:5s}: {len(d["y"]):>9,} windows, positive rate {d["y"].mean():.4f}, padded matrix {d["P"].nbytes / 1e9:.2f} GB')
print('Channels per time step:', N_CH)
DATA = DATA_BY_LEN[24]   # replaced by the selected window length after Step 10


class WindowSequence(keras.utils.PyDataset if hasattr(keras.utils, 'PyDataset') else keras.utils.Sequence):
    # Batches of (window, label).
    # Training mode: every pass = all positive hours + a fresh random sample of negative hours, shuffled.
    # One pass is split into `chunks_per_pass` Keras "epochs", so validation runs several times per pass.

    def __init__(self, d, batch_size, train=False, neg_frac=1.0, seed=0, with_labels=True, chunks_per_pass=1, **kwargs):
        try:
            super().__init__(**kwargs)
        except TypeError:
            super().__init__()
        self.P, self.starts, self.y = d['P'], d['starts'], d['y']
        self.seq_len = d['seq_len']
        self.bs, self.train, self.neg_frac, self.with_labels = batch_size, train, neg_frac, with_labels
        self.chunks = max(1, chunks_per_pass) if train else 1
        self.chunk_i = 0
        self.rng = np.random.default_rng(seed)
        self.offsets = np.arange(self.seq_len)[None, :]
        self.pos_idx = np.where(self.y == 1)[0]
        self.neg_idx = np.where(self.y == 0)[0]
        self._resample()

    def _resample(self):
        if self.train:
            n_neg = int(len(self.neg_idx) * self.neg_frac)
            neg = self.neg_idx if n_neg >= len(self.neg_idx) else self.rng.choice(self.neg_idx, n_neg, replace=False)
            idx = np.concatenate([self.pos_idx, neg])
            self.rng.shuffle(idx)
            self.idx = idx
        else:
            self.idx = np.arange(len(self.y))
        self.batches_per_pass = math.ceil(len(self.idx) / self.bs)
        self.chunk_len = math.ceil(self.batches_per_pass / self.chunks)

    def __len__(self):
        return self.chunk_len

    def __getitem__(self, b):
        g = min(self.chunk_i * self.chunk_len + b, self.batches_per_pass - 1)
        ids = self.idx[g * self.bs:(g + 1) * self.bs]
        xb = self.P[self.starts[ids][:, None] + self.offsets]
        return (xb, self.y[ids]) if self.with_labels else xb

    def on_epoch_end(self):
        if self.train:
            self.chunk_i += 1
            if self.chunk_i >= self.chunks:
                self.chunk_i = 0
                self._resample()


def train_prior(y, neg_frac):
    # Positive rate the network sees after negative subsampling (initialises the output bias).
    n_pos = float(y.sum())
    return n_pos / (n_pos + neg_frac * (len(y) - n_pos))

## 9. CNN-LSTM model
```
input (24 h × features)
 → feature dropout (drops whole input channels, robustness to missing variables)
 → causal Conv1D(128, k=3) → BN → ReLU
 → causal dilated Conv1D(128, k=3, d=2) → BN → ReLU      # local temporal patterns, receptive field 7 h
 → spatial dropout
 → LSTM(128) → LSTM(64)                                   # longer-range dynamics
 → [last hidden state ‖ attention-pooled hidden states]   # "now" + "what mattered in the last day"
 → Dense(64, ReLU) → Dropout(0.3) → Dense(1, sigmoid)
```
Layer widths above are for `size='base'` (Run 1). `size='small'` uses Conv1D(64) → LSTM(64) → LSTM(32) → Dense(32). Optional Gaussian input noise and the dropout rates are hyperparameters searched in Step 10.

*Causal* convolutions mean the output at hour *t* never looks at later hours. The LSTMs use the cuDNN-compatible configuration (no recurrent dropout), which keeps GPU training fast. Every layer is a built-in Keras layer, so the saved `.keras` file loads anywhere without custom code.

In [ ]:
SIZES = {'small': dict(conv=64, lstm1=64, lstm2=32, dense=32),
         'base': dict(conv=128, lstm1=128, lstm2=64, dense=64)}
# Run 1 settings - the reference configuration C0 in the search
BASE_HP = dict(seq_len=24, size='base', lr=1e-3, wd=1e-4, drop_in=0.1, drop_h=0.3, noise=0.0, neg_frac=0.3)


def build_cnn_lstm(seq_len, n_ch, prior, hp):
    w = SIZES[hp['size']]
    inp = keras.Input(shape=(seq_len, n_ch), name='window')
    x = layers.SpatialDropout1D(hp['drop_in'], name='feature_dropout')(inp)
    if hp['noise'] > 0:
        x = layers.GaussianNoise(hp['noise'], name='input_noise')(x)
    x = layers.Conv1D(w['conv'], 3, padding='causal', name='conv1')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv1D(w['conv'], 3, padding='causal', dilation_rate=2, name='conv2_dilated')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.SpatialDropout1D(min(0.5, hp['drop_h'] * 2 / 3))(x)    # 0.2 with Run 1 settings
    x = layers.LSTM(w['lstm1'], return_sequences=True, name='lstm1')(x)
    x = layers.LSTM(w['lstm2'], return_sequences=True, name='lstm2')(x)
    last = layers.Flatten(name='last_step')(layers.Cropping1D((seq_len - 1, 0))(x))
    scores = layers.Flatten()(layers.Dense(1, name='attention_score')(x))
    attn = layers.Reshape((seq_len, 1))(layers.Softmax(name='attention_weights')(scores))
    weighted = layers.Multiply()([x, attn])                       # broadcast (L,H) * (L,1)
    context = layers.GlobalAveragePooling1D(name='attention_context')(weighted)   # = weighted sum / L
    h = layers.Concatenate()([last, context])
    h = layers.Dense(w['dense'], activation='relu')(h)
    h = layers.Dropout(hp['drop_h'])(h)
    out = layers.Dense(1, activation='sigmoid', name='sepsis_prob',
                       bias_initializer=keras.initializers.Constant(float(np.log(prior / (1 - prior)))))(h)
    return keras.Model(inp, out, name='cnn_lstm_sepsis')


_m = build_cnn_lstm(BASE_HP['seq_len'], N_CH, 0.05, BASE_HP)
_m.summary()
try:
    keras.utils.plot_model(_m, to_file=os.path.join(PLOT_DIR, '02_model_architecture.png'),
                           show_shapes=True, show_layer_names=True, dpi=80)
except Exception as e:
    print('plot_model skipped:', type(e).__name__)
del _m

## 10. Hyperparameter search (validation only), then final training
Run 1 overfitted after one pass over the data, so the search targets **slower, more regularised learning**. It is a small, hand-designed grid that is easy to describe in a methods section; each configuration changes one or two things relative to the previous one:

| Config | Change |
|---|---|
| C0 | Run 1 settings (reference): base network, LR 1e-3, weight decay 1e-4 |
| C1 / C2 | Learning rate 3e-4 / 1e-4, weight decay 1e-3 |
| C3 / C4 | Smaller network with LR 3e-4 / 1e-4 |
| C5 / C6 | Strong regularisation (more dropout + input noise), base / small network |
| C7 | C6, but every negative hour in every pass (no subsampling) |
| C8 / C9 | C6 / C5 with a 48-hour window |

Each configuration is trained once (seed 42), with validation 4× per pass, early stopping on **validation AUPRC**, and best weights restored. The configuration with the highest validation AUPRC is then retrained with 5 seeds to form the final ensemble. The loss is still plain binary cross-entropy; the probability correction for negative subsampling uses the selected `neg_frac`.

In [ ]:
class ValidationMetrics(keras.callbacks.Callback):
    # Exact sklearn AUROC/AUPRC on the full validation set at every check.

    def __init__(self, d, evals_per_pass, label=''):
        super().__init__()
        self.seq = WindowSequence(d, CFG['PRED_BATCH_SIZE'], with_labels=False)
        self.y, self.k, self.label = d['y'], evals_per_pass, label
        self.history = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        p = self.model.predict(self.seq, verbose=0).ravel()
        logs['val_auroc'] = roc_auc_score(self.y, p)
        logs['val_auprc'] = average_precision_score(self.y, p)
        lr = self.model.optimizer.learning_rate
        lr = float(lr.numpy() if hasattr(lr, 'numpy') else lr)
        self.history.append(dict(check=epoch + 1, passes=(epoch + 1) / self.k, loss=float(logs.get('loss', np.nan)),
                                 val_auroc=logs['val_auroc'], val_auprc=logs['val_auprc'], lr=lr))
        star = ' *' if logs['val_auprc'] >= max(h['val_auprc'] for h in self.history) else ''
        print(f'  {self.label:6s} pass {(epoch + 1) / self.k:5.2f}: loss {logs.get("loss", np.nan):.4f} | '
              f'val AUROC {logs["val_auroc"]:.4f} | val AUPRC {logs["val_auprc"]:.4f}{star} | lr {lr:.1e}')


def train_one(hp, seed, label=''):
    # Train one CNN-LSTM with hyperparameters hp; returns (model with best weights, history, summary).
    set_seed(seed)
    d = DATA_BY_LEN[int(hp['seq_len'])]
    prior = train_prior(d['train']['y'], hp['neg_frac'])
    model = build_cnn_lstm(int(hp['seq_len']), N_CH, prior, hp)
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=hp['lr'], weight_decay=hp['wd'], clipnorm=1.0),
                  loss='binary_crossentropy')
    train_seq = WindowSequence(d['train'], CFG['BATCH_SIZE'], train=True, neg_frac=hp['neg_frac'], seed=seed,
                               chunks_per_pass=CFG['EVALS_PER_PASS'], workers=CFG['WORKERS'])
    vm = ValidationMetrics(d['val'], CFG['EVALS_PER_PASS'], label)
    callbacks = [
        vm,   # must come first: it adds val_auprc to the logs used below
        keras.callbacks.ReduceLROnPlateau(monitor='val_auprc', mode='max', factor=0.5,
                                          patience=CFG['LR_PATIENCE'], min_lr=1e-6, verbose=0),
        keras.callbacks.EarlyStopping(monitor='val_auprc', mode='max', patience=CFG['PATIENCE'], restore_best_weights=True),
    ]
    t0 = time.time()
    model.fit(train_seq, epochs=CFG['MAX_PASSES'] * CFG['EVALS_PER_PASS'], callbacks=callbacks, verbose=0)
    hist = pd.DataFrame(vm.history)
    best = hist.loc[hist['val_auprc'].idxmax()]
    info = dict(best_pass=float(best['passes']), val_auroc=float(best['val_auroc']), val_auprc=float(best['val_auprc']),
                passes_run=float(hist['passes'].iloc[-1]), minutes=(time.time() - t0) / 60)
    del train_seq
    return model, hist, info

In [ ]:
SWEEP = [
    ('C0 Run-1 settings (reference)', {}),
    ('C1 LR 3e-4, wd 1e-3',           dict(lr=3e-4, wd=1e-3)),
    ('C2 LR 1e-4, wd 1e-3',           dict(lr=1e-4, wd=1e-3)),
    ('C3 small net, LR 3e-4',         dict(size='small', lr=3e-4, wd=1e-3)),
    ('C4 small net, LR 1e-4',         dict(size='small', lr=1e-4, wd=1e-3)),
    ('C5 base, strong reg.',          dict(lr=3e-4, wd=1e-3, drop_in=0.2, drop_h=0.5, noise=0.1)),
    ('C6 small, strong reg.',         dict(size='small', lr=3e-4, wd=1e-3, drop_in=0.2, drop_h=0.5, noise=0.1)),
    ('C7 C6 + all negatives',         dict(size='small', lr=3e-4, wd=1e-3, drop_in=0.2, drop_h=0.5, noise=0.1, neg_frac=1.0)),
    ('C8 C6 + 48-h window',           dict(seq_len=48, size='small', lr=3e-4, wd=1e-3, drop_in=0.2, drop_h=0.5, noise=0.1)),
    ('C9 C5 + 48-h window',           dict(seq_len=48, lr=3e-4, wd=1e-3, drop_in=0.2, drop_h=0.5, noise=0.1)),
]
if SMOKE_TEST:
    SWEEP = [SWEEP[0], SWEEP[8]]

# To skip the search (e.g. re-running after a crash), set RUN_SWEEP=False in CFG and put the winner here:
BEST_HP_OVERRIDE = None   # e.g. dict(BASE_HP, size='small', lr=3e-4, wd=1e-3, drop_in=0.2, drop_h=0.5, noise=0.1)


def as_python(v):
    return v.item() if hasattr(v, 'item') else v


sweep_rows, sweep_hist = [], []
if CFG['RUN_SWEEP'] and BEST_HP_OVERRIDE is None:
    t_sweep = time.time()
    for name, change in SWEEP:
        hp = {**BASE_HP, **change}
        tag = name.split()[0]
        print(f'\n--- {name}: {hp}')
        model, hist, info = train_one(hp, CFG['SEED'], tag)
        print(f'    {tag} best at pass {info["best_pass"]:.2f}: val AUROC {info["val_auroc"]:.4f}, '
              f'val AUPRC {info["val_auprc"]:.4f}  ({info["minutes"]:.1f} min)')
        sweep_rows.append(dict(config=name, **hp, **info))
        sweep_hist.append(hist.assign(config=name))
        del model
        keras.backend.clear_session()
        gc.collect()
    SWEEP_RESULTS = pd.DataFrame(sweep_rows).sort_values('val_auprc', ascending=False).reset_index(drop=True)
    SWEEP_RESULTS.to_csv(os.path.join(ART_DIR, 'hyperparameter_search.csv'), index=False)
    pd.concat(sweep_hist).to_csv(os.path.join(ART_DIR, 'hyperparameter_search_history.csv'), index=False)
    BEST_HP = {k: as_python(SWEEP_RESULTS.iloc[0][k]) for k in BASE_HP}
    print(f'\nSearch finished in {(time.time() - t_sweep) / 60:.1f} min')
else:
    BEST_HP = dict(BEST_HP_OVERRIDE or BASE_HP)
    SWEEP_RESULTS = pd.DataFrame()
    print('Search skipped - using', BEST_HP)
BEST_HP['seq_len'] = int(BEST_HP['seq_len'])
print('Selected configuration (highest validation AUPRC):', BEST_HP)

In [ ]:
if len(SWEEP_RESULTS):
    fig, ax = plt.subplots(1, 2, figsize=(18, 5.5), gridspec_kw=dict(width_ratios=[1.4, 1]))
    for name, h in pd.concat(sweep_hist).groupby('config', sort=False):
        ax[0].plot(h['passes'], h['val_auprc'], 'o-', ms=3, label=name.split()[0],
                   lw=2.5 if name == SWEEP_RESULTS.config.iloc[0] else 1.2)
    ax[0].set(title='Validation AUPRC during training (each config, seed 42)', xlabel='passes over the training data',
              ylabel='validation AUPRC')
    ax[0].legend(ncol=2, fontsize=9)
    r = SWEEP_RESULTS.iloc[::-1]
    colors = ['#C44E52' if c.startswith('C0') else ('#55A868' if c == SWEEP_RESULTS.config.iloc[0] else '#4C72B0')
              for c in r.config]
    ax[1].barh(r.config, r.val_auprc, color=colors)
    ax[1].set_xlim(r.val_auprc.min() * 0.95, r.val_auprc.max() * 1.02)
    ax[1].set(title='Best validation AUPRC per configuration', xlabel='validation AUPRC')
    plt.tight_layout()
    savefig('03a_hyperparameter_search.png')
    display(SWEEP_RESULTS[['config', 'seq_len', 'size', 'lr', 'wd', 'drop_in', 'drop_h', 'noise', 'neg_frac',
                           'best_pass', 'val_auroc', 'val_auprc', 'minutes']].round(4))

In [ ]:
# Final ensemble: N_SEEDS models of the selected configuration
SEQ_LEN = BEST_HP['seq_len']
DATA = DATA_BY_LEN[SEQ_LEN]
for L in [L for L in DATA_BY_LEN if L != SEQ_LEN]:
    del DATA_BY_LEN[L]          # free the unused window length
gc.collect()
CALIB_LOG_OFFSET = float(np.log(BEST_HP['neg_frac']))

histories, val_preds_seeds, test_preds_seeds, model_paths = [], [], [], []
for k in range(CFG['N_SEEDS']):
    seed = CFG['SEED'] + 1000 * k
    print(f'\n===== Final CNN-LSTM seed {k + 1}/{CFG["N_SEEDS"]} (seed={seed}) =====')
    model, hist, info = train_one(BEST_HP, seed, f'seed{k + 1}')
    print(f'Seed {k + 1} done in {info["minutes"]:.1f} min - best at pass {info["best_pass"]:.2f}: '
          f'val AUROC {info["val_auroc"]:.4f}, val AUPRC {info["val_auprc"]:.4f}')
    path = os.path.join(MODEL_DIR, f'cnn_lstm_seed{k + 1}.keras')
    model.save(path)
    model_paths.append(path)
    histories.append(hist.assign(seed=k + 1))
    val_preds_seeds.append(sp.predict_windows([model], DATA['val']['P'], DATA['val']['starts'], SEQ_LEN, CFG['PRED_BATCH_SIZE']))
    test_preds_seeds.append(sp.predict_windows([model], DATA['test']['P'], DATA['test']['starts'], SEQ_LEN, CFG['PRED_BATCH_SIZE']))
    del model
    keras.backend.clear_session()
    gc.collect()

hist_df = pd.concat(histories)
hist_df.to_csv(os.path.join(ART_DIR, 'training_history.csv'), index=False)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for s, h in hist_df.groupby('seed'):
    ax[0].plot(h.passes, h.loss, 'o-', ms=3, label=f'seed {s}')
    ax[1].plot(h.passes, h.val_auroc, 'o-', ms=3, label=f'seed {s}')
    ax[2].plot(h.passes, h.val_auprc, 'o-', ms=3, label=f'seed {s}')
ax[0].set(title='Training loss (BCE, per chunk)', xlabel='passes over the training data')
ax[1].set(title='Validation AUROC', xlabel='passes over the training data')
ax[2].set(title='Validation AUPRC', xlabel='passes over the training data')
for a in ax:
    a.legend()
plt.tight_layout()
savefig('03_training_curves.png')

per_seed = pd.DataFrame([dict(seed=i + 1, val_auroc=roc_auc_score(DATA['val']['y'], p),
                              val_auprc=average_precision_score(DATA['val']['y'], p))
                         for i, p in enumerate(val_preds_seeds)])
p_ens = np.mean(val_preds_seeds, axis=0)
print(per_seed.round(4).to_string(index=False))
print(f'Ensemble of {len(val_preds_seeds)}: val AUROC {roc_auc_score(DATA["val"]["y"], p_ens):.4f}, '
      f'val AUPRC {average_precision_score(DATA["val"]["y"], p_ens):.4f}')
print('Run 1 ensemble for reference: val AUROC 0.8323, val AUPRC 0.1292')

## 11. Gradient-boosting baseline (LightGBM)
A strong tabular baseline trained on the same patients and the same engineered features (current hour only; trees handle missing values natively). Reviewers expect a comparison like this: it shows how much the sequence model adds on top of good features. Its predictions are also blended with the CNN-LSTM in Step 12.

In [ ]:
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    from sklearn.ensemble import HistGradientBoostingClassifier
print('LightGBM available:', HAS_LGB)

Xtr, ytr = X_all[masks['train']], DATA['train']['y']
Xva, yva = X_all[masks['val']], DATA['val']['y']
Xte = X_all[masks['test']]
t0 = time.time()
if HAS_LGB:
    gbm = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.05, num_leaves=63, min_child_samples=200,
                             subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0,
                             metric='average_precision', n_jobs=-1, random_state=CFG['SEED'], verbose=-1)
    gbm.fit(Xtr, ytr, eval_set=[(Xva, yva)],
            callbacks=[lgb.early_stopping(150, first_metric_only=True), lgb.log_evaluation(100)])
    gbm.booster_.save_model(os.path.join(MODEL_DIR, 'lightgbm_baseline.txt'))
else:
    gbm = HistGradientBoostingClassifier(max_iter=600, learning_rate=0.05, max_leaf_nodes=63,
                                         early_stopping=False, random_state=CFG['SEED'])
    gbm.fit(Xtr, ytr)
p_val_gbm = gbm.predict_proba(Xva)[:, 1]
p_test_gbm = gbm.predict_proba(Xte)[:, 1]
print(f'Baseline trained in {(time.time() - t0) / 60:.1f} min - val AUROC {roc_auc_score(yva, p_val_gbm):.4f}, '
      f'val AUPRC {average_precision_score(yva, p_val_gbm):.4f}')
del Xtr
gc.collect()

## 12. Probability calibration & blending (validation only)
Training on 30 % of the negatives inflates the CNN-LSTM's predicted odds by about 1/0.3. Adding `log(0.3)` to the logit restores the true event rate, and a reliability plot checks the result. The blend weight between the CNN-LSTM ensemble and LightGBM is the one that maximises **validation** AUPRC.

In [ ]:
yv, yt = DATA['val']['y'], DATA['test']['y']
P_VAL = {'CNN-LSTM': sp.calibrate(np.mean(val_preds_seeds, 0), CALIB_LOG_OFFSET), 'LightGBM': p_val_gbm}
P_TEST = {'CNN-LSTM': sp.calibrate(np.mean(test_preds_seeds, 0), CALIB_LOG_OFFSET), 'LightGBM': p_test_gbm}

weights = np.round(np.linspace(0, 1, 21), 2)
blend_scores = [average_precision_score(yv, w * P_VAL['CNN-LSTM'] + (1 - w) * P_VAL['LightGBM']) for w in weights]
W_BLEND = float(weights[int(np.argmax(blend_scores))])
P_VAL['Blend'] = W_BLEND * P_VAL['CNN-LSTM'] + (1 - W_BLEND) * P_VAL['LightGBM']
P_TEST['Blend'] = W_BLEND * P_TEST['CNN-LSTM'] + (1 - W_BLEND) * P_TEST['LightGBM']
print(f'Blend weight on CNN-LSTM chosen on validation: {W_BLEND}')

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(weights, blend_scores, 'o-')
ax[0].set(title='Validation AUPRC vs blend weight', xlabel='weight on CNN-LSTM', ylabel='AUPRC')
for name, p in P_VAL.items():
    frac, mean_pred = calibration_curve(yv, p, n_bins=15, strategy='quantile')
    ax[1].plot(mean_pred, frac, 'o-', label=f'{name} (Brier {brier_score_loss(yv, p):.4f})')
lim = max(ax[1].get_xlim()[1], ax[1].get_ylim()[1])
ax[1].plot([0, lim], [0, lim], 'k--', alpha=.5)
ax[1].set(title='Calibration (validation)', xlabel='mean predicted probability', ylabel='observed frequency')
ax[1].legend()
plt.tight_layout()
savefig('04_blend_and_calibration.png')

## 13. Threshold selection (validation only)
Three clinically motivated operating points are chosen **on validation**, then frozen:
* **Max F1**: balance of precision and recall at the hourly level.
* **Max utility**: the official PhysioNet 2019 utility score, which rewards alarms 12 h before to 3 h after onset and penalises late detection and false alarms.
* **Sensitivity ≥ 80 %**: a "don't miss sepsis" screening setting.

In [ ]:
util_val = sp.UtilityScorer(DATA['val']['meta'])
util_test = sp.UtilityScorer(DATA['test']['meta'])


def thr_max_f1(y, p):
    pr, rc, th = precision_recall_curve(y, p)
    f1 = 2 * pr[:-1] * rc[:-1] / (pr[:-1] + rc[:-1] + 1e-12)
    return float(th[np.argmax(f1)])


def thr_max_utility(scorer, p):
    cands = np.unique(np.quantile(p, np.linspace(0.0, 0.999, 400)))
    scores = [scorer.score(p >= t) for t in cands]
    return float(cands[int(np.argmax(scores))])


def thr_sensitivity(y, p, target):
    fpr, tpr, th = roc_curve(y, p)
    return float(th[np.argmax(tpr >= target)])


THRESHOLDS = {}
for name, p in P_VAL.items():
    THRESHOLDS[name] = {'max_f1': thr_max_f1(yv, p),
                        'max_utility': thr_max_utility(util_val, p),
                        f'sens_{int(CFG["TARGET_SENSITIVITY"] * 100)}': thr_sensitivity(yv, p, CFG['TARGET_SENSITIVITY'])}
pd.DataFrame(THRESHOLDS).T.round(4)

## 14. Final evaluation on the held-out test set
This is the first and only time the test patients are used. AUROC and AUPRC are threshold-free; everything else uses the thresholds frozen in Step 13. Confidence intervals come from resampling test *patients* (200 times); model comparisons use paired resamples. The AUPRC of a random classifier equals the positive rate (≈ 0.018), so read AUPRC against that baseline.

In [ ]:
def metrics_at(y, p, thr, scorer):
    pred = p >= thr
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if tp + fn else 0
    spec = tn / (tn + fp) if tn + fp else 0
    ppv = tp / (tp + fp) if tp + fp else 0
    npv = tn / (tn + fn) if tn + fn else 0
    f1 = 2 * ppv * sens / (ppv + sens) if ppv + sens else 0
    return dict(threshold=thr, sensitivity=sens, specificity=spec, precision_ppv=ppv, npv=npv, f1=f1,
                utility=scorer.score(pred), tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn))


rows = []
for name, p in P_TEST.items():
    base = dict(model=name, auroc=roc_auc_score(yt, p), auprc=average_precision_score(yt, p), brier=brier_score_loss(yt, p))
    for op, thr in THRESHOLDS[name].items():
        rows.append({**base, 'operating_point': op, **metrics_at(yt, p, thr, util_test)})
TEST_RESULTS = pd.DataFrame(rows)
TEST_RESULTS.to_csv(os.path.join(ART_DIR, 'test_results.csv'), index=False)
print(f'Test set: {len(yt):,} patient-hours from {len(SPLIT_IDS["test"]):,} patients, positive rate {yt.mean():.4f}')
TEST_RESULTS[['model', 'operating_point', 'auroc', 'auprc', 'sensitivity', 'specificity', 'precision_ppv',
              'f1', 'utility', 'threshold']].round(4)

In [ ]:
# patient-level bootstrap: 95% CIs for AUROC, AUPRC and utility, plus paired differences between models
test_codes = sp.patient_codes(DATA['test']['meta'])
rows_of = pd.Series(np.arange(len(test_codes))).groupby(test_codes).apply(np.array).to_list()
rng = np.random.default_rng(CFG['SEED'])
UTIL_OP = 'max_utility'
U1, U0, UB = util_test.u1, util_test.u0, util_test.best_row
pred_util = {n: P_TEST[n] >= THRESHOLDS[n][UTIL_OP] for n in P_TEST}


def utility_rows(pred, idx):
    """Normalised utility of a resample (utility is a sum over rows, so resampled patients just add up)."""
    u0 = U0[idx].sum()
    return (np.where(pred[idx], U1[idx], U0[idx]).sum() - u0) / (UB[idx].sum() - u0)


METRICS = ('auroc', 'auprc', 'utility')
boot = {n: {k: [] for k in METRICS} for n in P_TEST}
for _ in range(CFG['N_BOOTSTRAP']):
    pick = rng.integers(0, len(rows_of), len(rows_of))
    idx = np.concatenate([rows_of[i] for i in pick])
    if yt[idx].sum() == 0:
        continue
    for n, p in P_TEST.items():
        boot[n]['auroc'].append(roc_auc_score(yt[idx], p[idx]))
        boot[n]['auprc'].append(average_precision_score(yt[idx], p[idx]))
        boot[n]['utility'].append(utility_rows(pred_util[n], idx))
boot = {n: {k: np.asarray(v) for k, v in b.items()} for n, b in boot.items()}
point = {n: dict(auroc=roc_auc_score(yt, p), auprc=average_precision_score(yt, p),
                 utility=util_test.score(pred_util[n])) for n, p in P_TEST.items()}


def fmt_ci(v, b):
    return f'{v:.3f} ({np.percentile(b, 2.5):.3f}-{np.percentile(b, 97.5):.3f})'


CI = pd.DataFrame([{'model': n, 'AUROC': fmt_ci(point[n]['auroc'], b['auroc']),
                    'AUPRC': fmt_ci(point[n]['auprc'], b['auprc']),
                    f'Utility ({UTIL_OP} thr)': fmt_ci(point[n]['utility'], b['utility'])}
                   for n, b in boot.items()])
CI.to_csv(os.path.join(ART_DIR, 'test_auc_confidence_intervals.csv'), index=False)

# paired differences: every model is evaluated on the SAME resampled patients
REF = 'LightGBM'
diff_rows = []
for n in P_TEST:
    if n == REF:
        continue
    for k in METRICS:
        d = boot[n][k] - boot[REF][k]
        diff_rows.append(dict(comparison=f'{n} - {REF}', metric=k, difference=point[n][k] - point[REF][k],
                              ci_low=np.percentile(d, 2.5), ci_high=np.percentile(d, 97.5),
                              p_bootstrap_two_sided=min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))))
DIFF = pd.DataFrame(diff_rows)
DIFF.to_csv(os.path.join(ART_DIR, 'test_paired_differences.csv'), index=False)
print(f'Test-set metrics with 95% bootstrap CI ({len(boot[REF]["auroc"])} patient-level resamples):')
display(CI)
print('Paired bootstrap differences (95% CI excluding 0 = significant difference):')
DIFF.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
for name, p in P_TEST.items():
    fpr, tpr, _ = roc_curve(yt, p)
    ax[0].plot(fpr, tpr, label=f'{name} (AUROC {roc_auc_score(yt, p):.3f})')
    pr, rc, _ = precision_recall_curve(yt, p)
    ax[1].plot(rc, pr, label=f'{name} (AUPRC {average_precision_score(yt, p):.3f})')
ax[0].plot([0, 1], [0, 1], 'k--', alpha=.5)
ax[0].set(title='ROC curve - test set', xlabel='False positive rate (1 - specificity)', ylabel='Sensitivity')
ax[1].axhline(yt.mean(), color='k', ls='--', alpha=.5, label=f'No-skill ({yt.mean():.3f})')
ax[1].set(title='Precision-recall curve - test set', xlabel='Recall (sensitivity)', ylabel='Precision (PPV)')
for a in ax:
    a.legend(loc='best')
plt.tight_layout()
savefig('05_test_roc_pr.png')

PRIMARY = 'CNN-LSTM'
fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
for a, (op, thr) in zip(ax, THRESHOLDS[PRIMARY].items()):
    cm = confusion_matrix(yt, P_TEST[PRIMARY] >= thr)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', cbar=False, ax=a,
                xticklabels=['No sepsis', 'Sepsis'], yticklabels=['No sepsis', 'Sepsis'])
    a.set(title=f'{PRIMARY} - {op} (thr {thr:.3f})', xlabel='Predicted', ylabel='Actual')
plt.tight_layout()
savefig('06_test_confusion_matrices.png')

plt.figure(figsize=(9, 4.5))
for lbl, c in ((0, '#4C72B0'), (1, '#C44E52')):
    plt.hist(P_TEST[PRIMARY][yt == lbl], bins=60, alpha=.6, density=True, color=c,
             label=['No sepsis', 'Sepsis (label=1)'][lbl])
plt.yscale('log')
plt.xlabel('Predicted probability')
plt.ylabel('density (log)')
plt.title(f'{PRIMARY} - test-set probability distribution')
plt.legend()
savefig('07_test_probability_distribution.png')

## 15. Early-warning analysis (patient level)
Hourly metrics undersell a clinical alarm system. With clinical onset t_sepsis = first positive label + 6 h, a septic patient counts as **detected** if an alarm fires inside the clinically attributable window from 48 h before to 3 h after onset. The **lead time** is the number of hours between the first alarm in that window and onset. Alarms far earlier than this are not credited as detections. The table also reports how many non-septic patients ever trigger an alarm, and the false-alarm burden as alarm-hours per non-septic patient-day (hours with the alarm on, not separate alarm events).

Note that septic stays in this dataset end at most 9 h after the first positive label and 15 % of septic patients are already positive at their first ICU hour, so the 48-h window usually covers the whole septic stay: read the detection rate together with the false-alarm columns, not on its own.

In [ ]:
EW_LOOKBACK_H = 48


def early_warning(meta, p, thr):
    d = meta[['Patient_ID', 'hour_idx', 'SepsisLabel']].copy()
    d['alarm'] = p >= thr
    out = []
    for pid, g in d.groupby('Patient_ID', sort=False):
        lab, alarm, h = g.SepsisLabel.to_numpy(), g.alarm.to_numpy(), g.hour_idx.to_numpy()
        if lab.any():
            t_sepsis = h[lab == 1][0] + 6
            in_win = alarm & (h >= t_sepsis - EW_LOOKBACK_H) & (h <= t_sepsis + 3)
            first = h[in_win][0] if in_win.any() else np.nan
            out.append(dict(septic=True, detected=bool(in_win.any()), hours_before_onset=t_sepsis - first))
        else:
            out.append(dict(septic=False, any_false_alarm=bool(alarm.any()), alarm_hours=int(alarm.sum()), stay_hours=len(g)))
    return pd.DataFrame(out)


ew_rows, ew_detail = [], {}
for op, thr in THRESHOLDS[PRIMARY].items():
    ew = early_warning(DATA['test']['meta'], P_TEST[PRIMARY], thr)
    sep, non = ew[ew.septic], ew[~ew.septic]
    det = sep[sep.detected]
    ew_detail[op] = det
    ew_rows.append(dict(operating_point=op, threshold=thr,
                        septic_patients=len(sep), detected=f'{det.shape[0]} ({len(det) / max(len(sep), 1):.1%})',
                        median_hours_before_onset=det.hours_before_onset.median(),
                        detected_6h_or_more_before=f'{(det.hours_before_onset >= 6).mean():.1%}' if len(det) else 'n/a',
                        detected_any_time_before_onset=f'{(det.hours_before_onset > 0).mean():.1%}' if len(det) else 'n/a',
                        non_septic_patients=len(non),
                        with_any_false_alarm=f'{non.any_false_alarm.mean():.1%}',
                        alarm_hours_per_non_septic_patient_day=round(non.alarm_hours.sum() / max(non.stay_hours.sum(), 1) * 24, 2)))
EARLY_WARNING = pd.DataFrame(ew_rows)
EARLY_WARNING.to_csv(os.path.join(ART_DIR, 'test_early_warning.csv'), index=False)

op = 'max_utility'
plt.figure(figsize=(9, 4.5))
plt.hist(ew_detail[op].hours_before_onset, bins=np.arange(-3.5, EW_LOOKBACK_H + 1.5, 1), color='#8172B2')
plt.axvline(6, color='k', ls='--', label='6 h before onset')
plt.xlabel('Hours between first alarm and clinical onset (positive = earlier)')
plt.ylabel('Septic patients detected')
plt.title(f'{PRIMARY} - timing of first alarm ({op} threshold, test set)')
plt.legend()
savefig('08_early_warning_timing.png')
EARLY_WARNING

## 16. Interpretability
* **CNN-LSTM, grouped permutation importance**: all features derived from one clinical variable (e.g. every `Lactate*` feature) are shuffled together across patients over the full input window, and the drop in validation AUPRC is measured. It is model-agnostic and works directly on the sequence model.
* **LightGBM, SHAP (TreeExplainer)**: exact Shapley values per feature, which are useful for per-patient explanations in the web app.

In [ ]:
cnn_models = [keras.models.load_model(p) for p in model_paths]
rng = np.random.default_rng(CFG['SEED'])
yv_all = DATA['val']['y']
pos = np.where(yv_all == 1)[0]
neg = np.where(yv_all == 0)[0]
sub = np.concatenate([rng.choice(pos, min(len(pos), 4000), replace=False),
                      rng.choice(neg, min(len(neg), 16000), replace=False)])
Wv = sp.gather_windows(DATA['val']['P'], DATA['val']['starts'][sub], SEQ_LEN)
ysub = yv_all[sub]


def ens_predict(W):
    return np.mean([m.predict(W, batch_size=CFG['PRED_BATCH_SIZE'], verbose=0).ravel() for m in cnn_models], 0)


base_auprc = average_precision_score(ysub, ens_predict(Wv))
groups = {}
for j, f in enumerate(FEATURES):
    groups.setdefault(sp.feature_group(f), []).append(j)
imp = []
for g, cols in groups.items():
    Wp = Wv.copy()
    perm = rng.permutation(len(Wp))
    Wp[:, :, cols] = Wv[:, :, cols][perm]
    imp.append(dict(variable=g, n_features=len(cols), auprc_drop=base_auprc - average_precision_score(ysub, ens_predict(Wp))))
    del Wp
PERM_IMP = pd.DataFrame(imp).sort_values('auprc_drop', ascending=False)
PERM_IMP.to_csv(os.path.join(ART_DIR, 'cnn_lstm_permutation_importance.csv'), index=False)

top = PERM_IMP.head(25).iloc[::-1]
plt.figure(figsize=(9, 8))
plt.barh(top.variable, top.auprc_drop, color='#4C72B0')
plt.xlabel('Drop in validation AUPRC when shuffled')
plt.title('CNN-LSTM - grouped permutation importance (top 25)')
savefig('09_cnn_lstm_permutation_importance.png')
del Wv
gc.collect()

In [ ]:
try:
    import shap
    rng = np.random.default_rng(CFG['SEED'])
    te_idx = np.where(masks['test'])[0]
    yte_ = DATA['test']['y']
    pick = np.concatenate([rng.choice(np.where(yte_ == 1)[0], min(1000, int(yte_.sum())), replace=False),
                           rng.choice(np.where(yte_ == 0)[0], 2000, replace=False)])
    X_shap = X_all.iloc[te_idx[pick]]
    explainer = shap.TreeExplainer(gbm.booster_ if HAS_LGB else gbm)
    sv = explainer.shap_values(X_shap)
    sv = sv[1] if isinstance(sv, list) else sv
    shap.summary_plot(sv, X_shap, max_display=25, show=False)
    plt.title('LightGBM - SHAP summary (test sample)')
    savefig('10_lightgbm_shap_beeswarm.png')
    shap.summary_plot(sv, X_shap, plot_type='bar', max_display=25, show=False)
    plt.title('LightGBM - mean |SHAP|')
    savefig('11_lightgbm_shap_bar.png')
    pd.DataFrame({'feature': X_shap.columns, 'mean_abs_shap': np.abs(sv).mean(0)}) \
      .sort_values('mean_abs_shap', ascending=False).to_csv(os.path.join(ART_DIR, 'lightgbm_shap_importance.csv'), index=False)
except Exception as e:
    print('SHAP step skipped:', type(e).__name__, e)

## 17. Export artifacts & reload check
Everything needed to reproduce the results, test again later, or deploy:

| File | Purpose |
|---|---|
| `test_patients_raw.csv` | **The held-out test patients in the original hourly format.** Never used for training, tuning or thresholds |
| `test_predictions.csv` | Per-hour test predictions of every model, with the label |
| `models/cnn_lstm_seed*.keras` | The trained CNN-LSTM ensemble members |
| `models/lightgbm_baseline.txt` | LightGBM baseline |
| `preprocessor.json` | Clipping bounds, means and SDs (fitted on training patients) |
| `model_config.json` | Sequence length, selected hyperparameters, calibration offset, blend weight, thresholds, test metrics |
| `hyperparameter_search*.csv` | Every configuration tried in Step 10 and its validation curve |
| `sepsis_pipeline.py` | Feature engineering + inference code (`predict_raw`) for the backend |
| `split_patient_ids.json` | Exact patient IDs in train / val / test |
| `*.csv`, `plots/` | All tables and figures for the paper |

In [ ]:
test_out = DATA['test']['meta'][['Patient_ID', 'ICULOS', 'hour_idx', 'SepsisLabel']].copy()
for name, p in P_TEST.items():
    col = name.lower().replace('-', '_')
    test_out[f'p_{col}'] = p
    for op, thr in THRESHOLDS[name].items():
        test_out[f'alarm_{col}_{op}'] = (p >= thr).astype(np.int8)
test_out.to_csv(os.path.join(ART_DIR, 'test_predictions.csv'), index=False)

primary_rows = TEST_RESULTS[TEST_RESULTS.model == PRIMARY]
MODEL_CONFIG = dict(
    created=time.strftime('%Y-%m-%d %H:%M'),
    smoke_test=SMOKE_TEST,
    seq_len=SEQ_LEN,
    cnn_lstm_hparams=BEST_HP,
    hyperparameter_search=SWEEP_RESULTS.to_dict(orient='records'),
    n_channels=N_CH,
    features=FEATURES,
    calibration_log_offset=CALIB_LOG_OFFSET,
    blend_weight_cnn_lstm=W_BLEND,
    cnn_lstm_models=[os.path.relpath(p, ART_DIR) for p in model_paths],
    thresholds=THRESHOLDS,
    primary_model=PRIMARY,
    config=CFG,
    test_metrics=dict(auroc=float(primary_rows.auroc.iloc[0]), auprc=float(primary_rows.auprc.iloc[0]),
                      by_operating_point=primary_rows.drop(columns=['model']).to_dict(orient='records')),
    versions=dict(tensorflow=tf.__version__, keras=keras.__version__, numpy=np.__version__, pandas=pd.__version__),
)
with open(os.path.join(ART_DIR, 'model_config.json'), 'w') as f:
    json.dump(MODEL_CONFIG, f, indent=2, default=float)
shutil.copy('sepsis_pipeline.py', os.path.join(ART_DIR, 'sepsis_pipeline.py'))
print('Saved predictions, config and pipeline module.')

In [ ]:
# Reload everything from disk, as the backend would, and re-score the raw test file.
importlib.reload(sp)
cfg = json.load(open(os.path.join(ART_DIR, 'model_config.json')))
prep_loaded = sp.Preprocessor.load(os.path.join(ART_DIR, 'preprocessor.json'))
models_loaded = [keras.models.load_model(os.path.join(ART_DIR, p)) for p in cfg['cnn_lstm_models']]
raw_test = pd.read_csv(os.path.join(ART_DIR, 'test_patients_raw.csv'), dtype={'Patient_ID': str})
rescored = sp.predict_raw(raw_test, models_loaded, prep_loaded, cfg)

assert (rescored['Patient_ID'].to_numpy() == test_out['Patient_ID'].to_numpy()).all()
max_diff = np.abs(rescored['p_cnn_lstm'].to_numpy() - test_out['p_cnn_lstm'].to_numpy()).max()
print(f'Re-scored {len(rescored):,} test hours from the exported files. Max probability difference: {max_diff:.2e}')
assert max_diff < 1e-3, 'Exported pipeline does not reproduce the notebook predictions'
print(f'Reload check PASSED - re-scored test AUROC {roc_auc_score(rescored.SepsisLabel, rescored.p_cnn_lstm):.4f}')

In [ ]:
zip_path = shutil.make_archive(os.path.join(WORK_DIR, 'sepsis_artifacts'), 'zip', ART_DIR)
print('Download this file:', zip_path, f'({os.path.getsize(zip_path) / 1e6:.0f} MB)')
for root, _, files in os.walk(ART_DIR):
    for fn in sorted(files):
        p = os.path.join(root, fn)
        print(f'  {os.path.relpath(p, ART_DIR):55s} {os.path.getsize(p) / 1e6:8.2f} MB')

## Summary for the paper

In [ ]:
print('=' * 90)
print('DATA')
print(split_summary.round(4).to_string(index=False))
print('\nTEST SET - threshold-free metrics (95% CI, patient-level bootstrap)')
print(CI.to_string(index=False))
print('\nTEST SET - paired bootstrap differences vs LightGBM')
print(DIFF.round(4).to_string(index=False))
print('\nTEST SET - operating points (thresholds chosen on validation)')
print(TEST_RESULTS[['model', 'operating_point', 'sensitivity', 'specificity', 'precision_ppv', 'f1', 'utility']]
      .round(3).to_string(index=False))
print(f'\nEARLY WARNING - {PRIMARY}')
print(EARLY_WARNING.drop(columns=['threshold']).to_string(index=False))
print('\nTOP 10 VARIABLES (CNN-LSTM permutation importance)')
print(PERM_IMP.head(10).round(4).to_string(index=False))

print('\nSELECTED CNN-LSTM CONFIGURATION (validation AUPRC):', BEST_HP)
RUN1_TEST = pd.DataFrame({'model': ['CNN-LSTM', 'LightGBM', 'Blend'],
                          'auroc': [0.826, 0.840, 0.838], 'auprc': [0.116, 0.117, 0.123], 'utility': [0.384, 0.416, 0.415]})
run2 = pd.DataFrame([dict(model=n, auroc=point[n]['auroc'], auprc=point[n]['auprc'], utility=point[n]['utility']) for n in P_TEST])
cmp = RUN1_TEST.merge(run2, on='model', suffixes=('_run1', '_run2'))
for k in ('auroc', 'auprc', 'utility'):
    cmp[f'{k}_change'] = cmp[f'{k}_run2'] - cmp[f'{k}_run1']
if SPLIT_MATCHES_RUN1 and not SMOKE_TEST:
    print('\nRUN 1 (untuned) vs RUN 2 (tuned) - same test patients')
    print(cmp.round(3).to_string(index=False))
    cmp.to_csv(os.path.join(ART_DIR, 'run1_vs_run2_test.csv'), index=False)
else:
    print('\n(Run 1 vs Run 2 comparison not printed: different split or smoke test)')
print('=' * 90)

### Reporting checklist
* **Report** AUROC and AUPRC with 95 % CIs, sensitivity/specificity/PPV/F1 at the validation-chosen thresholds, the PhysioNet utility score, and the early-warning table (detection rate, hours before onset, alarm-hours per non-septic patient-day).
* **Describe the tuning** in the methods: the configurations in `hyperparameter_search.csv` were compared on validation AUPRC only, and the test set was used once.
* **Compare models** with the paired bootstrap differences (`test_paired_differences.csv`), not by checking whether two separate CIs overlap.
* **Compare** the CNN-LSTM against the LightGBM baseline honestly. "The sequence model matches / exceeds / is complementary to a strong tabular baseline" is a legitimate finding.
* **State** that the split is by patient, all preprocessing was fitted on training patients, thresholds were tuned on validation only, and no feature uses future data (Step 6).
* **Avoid** reporting accuracy (≈ 98 % even for a model that never predicts sepsis) or any "validation AUC" from the earlier notebooks.
* **Expect** hourly-level F1 well below 0.6 on this dataset. Published PhysioNet 2019 work typically reports AUROC of about 0.80–0.88 with low hourly precision. The patient-level early-warning numbers are the clinically meaningful ones.